In [2]:
# %% Cell 1: 2DOF RR 机械臂运动学建模（SymPy）
import sympy as sp
import numpy as np

# ── 符号定义 ──────────────────────────────────────────────
L1, L2 = sp.symbols("L1 L2", positive=True)       # 连杆长度
q1, q2 = sp.symbols("q1 q2", real=True)           # 关节角 θ₁, θ₂
x, y  = sp.symbols("x y", real=True)              # 末端笛卡尔坐标

# ══════════════════════════════════════════════════════════
# 正向运动学 (Forward Kinematics)
# ══════════════════════════════════════════════════════════
fx = L1 * sp.cos(q1) + L2 * sp.cos(q1 + q2)
fy = L1 * sp.sin(q1) + L2 * sp.sin(q1 + q2)

sp.Eq(sp.Matrix([x, y]), sp.Matrix([fx, fy]), evaluate=False)

# ── 数值化正向运动学 ─────────────────────────────────────
_fk = sp.lambdify([q1, q2, L1, L2], [fx, fy], "numpy")

def fk(q1_val, q2_val, L1_val=1.5, L2_val=1.0):
    """正向运动学：给定 θ₁,θ₂ → (x, y)"""
    xv, yv = _fk(q1_val, q2_val, L1_val, L2_val)
    return float(xv), float(yv)

# ══════════════════════════════════════════════════════════
# 逆向运动学 (Inverse Kinematics — elbow-up: θ₂ < 0)
# ══════════════════════════════════════════════════════════

# cos(θ₂) = (x² + y² - L1² - L2²) / (2 L1 L2)
c2 = (x**2 + y**2 - L1**2 - L2**2) / (2 * L1 * L2)

# elbow-up → sin(θ₂) < 0, 因此 θ₂ = -acos(c2)
th2_ik = -sp.acos(c2)

# θ₁ = atan2(y, x) - atan2(L2 sinθ₂, L1 + L2 cosθ₂)
th1_ik = sp.atan2(y, x) - sp.atan2(L2 * sp.sin(th2_ik), L1 + L2 * sp.cos(th2_ik))

# 简化表达式
th1_ik_s = sp.simplify(th1_ik)

sp.Eq(sp.Matrix([q1, q2]), sp.Matrix([th1_ik_s, th2_ik]), evaluate=False)

# ── 数值化逆向运动学 ─────────────────────────────────────
_ik = sp.lambdify([x, y, L1, L2], [th1_ik_s, th2_ik], "numpy")

def ik(x_val, y_val, L1_val=1.5, L2_val=1.0):
    """逆向运动学 (elbow-up)：给定 (x,y) → (θ₁, θ₂)"""
    q1v, q2v = _ik(x_val, y_val, L1_val, L2_val)
    return float(q1v), float(q2v)

# ══════════════════════════════════════════════════════════
# 验证 (round-trip)
# ══════════════════════════════════════════════════════════
L1_num, L2_num = 1.5, 1.0
test_cases = [
    (0.4, 1.2),
    (0.8, -0.6),
    (1.6, 0.4),
    (-1.0, 1.8),
]

print(f"连杆: L1={L1_num}, L2={L2_num}")
print(f"{'目标(x,y)':>14s}  {'ik→(θ₁,θ₂)':>20s}  {'fk→(x,y)':>18s}  {'误差':>10s}")
print("-" * 70)
for tx, ty in test_cases:
    q1_ik, q2_ik = ik(tx, ty, L1_num, L2_num)
    x_fk, y_fk = fk(q1_ik, q2_ik, L1_num, L2_num)
    err = np.hypot(tx - x_fk, ty - y_fk)
    print(f"  ({tx:5.2f},{ty:5.2f})    ({q1_ik:7.4f},{q2_ik:7.4f})    ({x_fk:6.4f},{y_fk:6.4f})    {err:.2e}")
print()

# 检查 elbow-up 约束: θ₂ < 0
for tx, ty in test_cases:
    q1_ik, q2_ik = ik(tx, ty, L1_num, L2_num)
    assert q2_ik < 0, f"elbow-up violated at ({tx},{ty}): θ₂={q2_ik:.3f}"
print("✓ 所有测试点满足 elbow-up (θ₂ < 0) 约束")

连杆: L1=1.5, L2=1.0
       目标(x,y)            ik→(θ₁,θ₂)            fk→(x,y)          误差
----------------------------------------------------------------------
  ( 0.40, 1.20)    ( 1.9702,-2.1532)    (0.4000,1.2000)    1.11e-16
  ( 0.80,-0.60)    ( 0.0792,-2.4189)    (0.8000,-0.6000)    1.57e-16
  ( 1.60, 0.40)    ( 0.8845,-1.7484)    (1.6000,0.4000)    1.11e-16
  (-1.00, 1.80)    ( 2.5541,-1.2345)    (-1.0000,1.8000)    4.00e-16

✓ 所有测试点满足 elbow-up (θ₂ < 0) 约束


In [3]:
# %% Cell 2: 交互式可视化（matplotlib + ipywidgets）
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, FloatSlider, VBox, HBox, Label, Output
import numpy as np

# ── 复用 Cell 1 的函数 ────────────────────────────────────
L1_val = 1.5
L2_val = 1.0

# 工作空间范围（用于 visual 边界）
R_max = L1_val + L2_val
R_min = abs(L1_val - L2_val)

@interact(
    mode=[("正运动学 (调角度)", "fk"), ("逆运动学 (调坐标)", "ik")],
    q1=FloatSlider(value=0.8, min=-np.pi, max=np.pi, step=0.01,
                   description="θ₁ (rad)", continuous_update=True),
    q2=FloatSlider(value=-1.8, min=-np.pi, max=np.pi, step=0.01,
                   description="θ₂ (rad)", continuous_update=True),
    px=FloatSlider(value=1.2, min=-R_max, max=R_max, step=0.01,
                   description="X target", continuous_update=True),
    py=FloatSlider(value=1.0, min=-R_max, max=R_max, step=0.01,
                   description="Y target", continuous_update=True),
)
def viz(mode, q1, q2, px, py):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-R_max - 0.3, R_max + 0.3)
    ax.set_ylim(-R_max - 0.3, R_max + 0.3)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title("2DOF RR 机械臂 — " + ("正运动学" if mode == "fk" else "逆运动学 (elbow-up)"))

    # ── 工作空间边界圆 ─────────────────────────────────
    workspace_outer = patches.Arc((0, 0), 2*R_max, 2*R_max, angle=0, theta1=0, theta2=360,
                                  color="lightgray", lw=1, ls="--", label="最大范围")
    workspace_inner = patches.Arc((0, 0), 2*R_min, 2*R_min, angle=0, theta1=0, theta2=360,
                                  color="lightgray", lw=1, ls=":", label="最小范围")
    ax.add_patch(workspace_outer)
    ax.add_patch(workspace_inner)

    # ── 图标原点 ───────────────────────────────────────
    ax.plot(0, 0, "ko", markersize=5, zorder=5)
    ax.annotate("base", (0, 0), textcoords="offset points", xytext=(6, -14), fontsize=8)

    # ── 根据模式选择关节角 ─────────────────────────────
    if mode == "ik":
        try:
            q1_val, q2_val = ik(px, py, L1_val, L2_val)
            if np.isnan(q1_val) or np.isnan(q2_val):
                raise ValueError("unreachable")
        except Exception:
            ax.text(0, 0, "✗ 不可达", ha="center", va="center", fontsize=16, color="red")
            plt.show()
            return
    else:
        q1_val, q2_val = q1, q2
        px_val, py_val = fk(q1_val, q2_val, L1_val, L2_val)
        # 更新 px/py（仅显示用）
        px, py = px_val, py_val

    # ── 连杆端点 ───────────────────────────────────────
    j1 = np.array([L1_val * np.cos(q1_val), L1_val * np.sin(q1_val)])
    ee = np.array([
        L1_val * np.cos(q1_val) + L2_val * np.cos(q1_val + q2_val),
        L1_val * np.sin(q1_val) + L2_val * np.sin(q1_val + q2_val),
    ])

    # ── 绘制连杆 ───────────────────────────────────────
    ax.plot([0, j1[0]], [0, j1[1]], "b-", lw=3, label=f"Link 1 ({L1_val}m)")
    ax.plot([j1[0], ee[0]], [j1[1], ee[1]], "r-", lw=3, label=f"Link 2 ({L2_val}m)")

    # ── 关节点 ─────────────────────────────────────────
    ax.plot(j1[0], j1[1], "ko", markersize=7, zorder=5)
    ax.annotate("J1", j1, textcoords="offset points", xytext=(6, 6), fontsize=8)

    # ── 末端执行器 ─────────────────────────────────────
    ax.plot(ee[0], ee[1], "go", markersize=10, zorder=5, markeredgecolor="black", markeredgewidth=1)
    ax.annotate("EE", ee, textcoords="offset points", xytext=(6, 6), fontsize=9, color="green")

    # ── 角度弧（示意） ─────────────────────────────────
    arc_radius = 0.25
    if abs(q1_val) > 0.02:
        arc_q1 = patches.Arc((0, 0), 2*arc_radius, 2*arc_radius, angle=0,
                              theta1=0, theta2=np.degrees(q1_val),
                              color="blue", lw=1.5, fill=False)
        ax.add_patch(arc_q1)
    if abs(q2_val) > 0.02:
        arc_q2 = patches.Arc(j1, 2*arc_radius, 2*arc_radius, angle=np.degrees(q1_val),
                              theta1=0, theta2=np.degrees(q2_val),
                              color="red", lw=1.5, fill=False)
        ax.add_patch(arc_q2)

    # ── 信息面板 ───────────────────────────────────────
    info = (
        f"θ₁ = {np.degrees(q1_val):+.1f}°  ({q1_val:+.4f} rad)\n"
        f"θ₂ = {np.degrees(q2_val):+.1f}°  ({q2_val:+.4f} rad)\n"
        f"elbow-up: {'✓' if q2_val < 0 else '✗ (elbow-down!)'}\n"
        f"EE  = ({ee[0]:.4f}, {ee[1]:.4f})"
    )
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=9,
            verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85))

    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()

/home/lunarifish/engineer-amorph/.venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


interactive(children=(Dropdown(description='mode', options=(('正运动学 (调角度)', 'fk'), ('逆运动学 (调坐标)', 'ik')), value…